# Build Modules
This notebook creates all the modular `.py` files for the skincare recommendation pipeline.
Run all cells to create/update the files in the Google Drive shared folder.

**Structure created:**
```
skincare_project/
├── agents/
│   ├── __init__.py
│   ├── skin_profile.py
│   ├── retrieval.py
│   ├── conflict_checker.py
│   └── routine_builder.py
├── pipeline.py
└── data_loader.py
```

## Setup: Mount Drive & Create Folders

In [11]:
import os
from google.colab import drive
drive.mount('/content/drive')


# Create agents folder if it doesn't exist
BASE_DIR = "/content/drive/MyDrive/skincare_project"
os.makedirs(f"{BASE_DIR}/agents", exist_ok=True)
print("Folders ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready.


## agents/__init__.py
Makes the agents folder a proper Python package so we can import from it.

In [12]:
%%writefile /content/drive/MyDrive/skincare_project/agents/__init__.py
# Skincare Recommendation System - Agents Package

Overwriting /content/drive/MyDrive/skincare_project/agents/__init__.py


## data_loader.py
Handles all data loading, cleaning, merging, and ChromaDB indexing.

In [13]:
%%writefile /content/drive/MyDrive/skincare_project/data_loader.py
import pandas as pd
import ast
import chromadb

DATA_DIR = "/content/drive/MyDrive/skincare_project/data"


def parse_ingredients_list(ingred_str):
    """Parse ingredient list from string representation of Python list."""
    try:
        ingredients = ast.literal_eval(ingred_str)
        return ", ".join([i.strip().lower() for i in ingredients])
    except (ValueError, SyntaxError, TypeError):
        return ""


def extract_brand(name):
    """Extract brand name from product name for lookfantastic products."""
    brand_mapping = {
        "The Ordinary": "The Ordinary",
        "CeraVe": "CeraVe",
        "AMELIORATE": "AMELIORATE",
        "La Roche-Posay": "La Roche-Posay",
        "Peter Thomas Roth": "Peter Thomas Roth",
        "Paula's Choice": "Paula's Choice",
        "Drunk Elephant": "Drunk Elephant",
    }
    if pd.isna(name):
        return "Unknown"
    for brand in brand_mapping:
        if brand.lower() in name.lower():
            return brand_mapping[brand]
    parts = name.split(" ")
    return " ".join(parts[:2]) if len(parts) >= 2 else parts[0]


def clean_list_string(val):
    """Parse string representation of list from INCI dataset."""
    if pd.isna(val):
        return ""
    try:
        items = ast.literal_eval(val)
        return ", ".join([i.strip() for i in items if i.strip()])
    except:
        return str(val).strip()


def load_and_clean_data():
    """Load all datasets, clean, merge, and return unified dataframes."""

    # Load raw datasets
    skincare_df = pd.read_csv(f"{DATA_DIR}/skincare_products_clean.csv")
    cosmetics_df = pd.read_csv(f"{DATA_DIR}/cosmetics.csv")
    inci_df = pd.read_csv(f"{DATA_DIR}/ingredientsList.csv")

    # Clean skincare_products_clean
    skincare_df["ingredients_clean"] = skincare_df["clean_ingreds"].apply(parse_ingredients_list)
    skincare_df_clean = skincare_df[["product_name", "product_type", "ingredients_clean", "price"]].copy()
    skincare_df_clean.columns = ["name", "product_type", "ingredients", "price"]
    skincare_df_clean["brand"] = "Unknown"
    skincare_df_clean["source"] = "lookfantastic"
    skincare_df_clean["brand"] = skincare_df_clean["name"].apply(lambda x: x.split(" ")[0] if pd.notna(x) else "Unknown")
    for col in ["combination", "dry", "normal", "oily", "sensitive"]:
        skincare_df_clean[col] = None

    # Clean cosmetics dataset
    cosmetics_df["ingredients_clean"] = cosmetics_df["Ingredients"].apply(
        lambda x: ", ".join([i.strip().lower() for i in str(x).split(",")]) if pd.notna(x) else ""
    )
    cosmetics_df_clean = cosmetics_df[["Name", "Label", "ingredients_clean", "Price", "Brand",
                                        "Combination", "Dry", "Normal", "Oily", "Sensitive"]].copy()
    cosmetics_df_clean.columns = ["name", "product_type", "ingredients", "price", "brand",
                                   "combination", "dry", "normal", "oily", "sensitive"]
    cosmetics_df_clean["source"] = "sephora"

    # Merge
    products_df = pd.concat([skincare_df_clean, cosmetics_df_clean], ignore_index=True)

    # Fix brand extraction for lookfantastic products
    mask = products_df["source"] == "lookfantastic"
    products_df.loc[mask, "brand"] = products_df.loc[mask, "name"].apply(extract_brand)

    # Standardize product type names
    product_type_mapping = {
        "Moisturiser": "Moisturizer",
        "Eye Care": "Eye cream",
    }
    exclude_types = ["Body Wash", "Bath Salts", "Bath Oil"]
    products_df = products_df[~products_df["product_type"].isin(exclude_types)].reset_index(drop=True)
    products_df["product_type"] = products_df["product_type"].replace(product_type_mapping)
    products_df = products_df[products_df["ingredients"].str.len() > 0].reset_index(drop=True)

    # Clean INCI
    inci_df["good_for"] = inci_df["who_is_it_good_for"].apply(clean_list_string)
    inci_df["avoid_for"] = inci_df["who_should_avoid"].apply(clean_list_string)
    inci_df["full_description"] = inci_df.apply(lambda row:
        f"Ingredient: {row['name']}. "
        f"Description: {row['short_description'] if pd.notna(row['short_description']) else 'N/A'}. "
        f"Function: {row['what_does_it_do'] if pd.notna(row['what_does_it_do']) else 'N/A'}. "
        f"Good for: {row['good_for']}. "
        f"Avoid if: {row['avoid_for']}.",
        axis=1
    )

    print(f"Loaded {len(products_df)} products and {len(inci_df)} ingredients.")
    return products_df, inci_df


def build_vector_database(products_df, inci_df):
    """Create ChromaDB collections for products and ingredients."""

    chroma_client = chromadb.Client()

    # Product collection
    try:
        chroma_client.delete_collection("skincare_products")
    except:
        pass

    product_collection = chroma_client.create_collection(
        name="skincare_products",
        metadata={"hnsw:space": "cosine"}
    )

    product_documents = []
    product_metadata = []
    product_ids = []

    for idx, row in products_df.iterrows():
        skin_types = []
        for st in ["combination", "dry", "normal", "oily", "sensitive"]:
            if row.get(st) == 1:
                skin_types.append(st)
        skin_type_str = ", ".join(skin_types) if skin_types else "not specified"

        doc = (
            f"Product: {row['name']}. "
            f"Brand: {row['brand']}. "
            f"Type: {row['product_type']}. "
            f"Price: {row['price']}. "
            f"Suitable for skin types: {skin_type_str}. "
            f"Ingredients: {row['ingredients']}."
        )
        product_documents.append(doc)
        product_metadata.append({
            "name": str(row["name"]),
            "brand": str(row["brand"]),
            "product_type": str(row["product_type"]),
            "price": str(row["price"]),
            "source": str(row["source"]),
            "skin_types": skin_type_str
        })
        product_ids.append(f"product_{idx}")

    batch_size = 500
    for i in range(0, len(product_documents), batch_size):
        end = min(i + batch_size, len(product_documents))
        product_collection.add(
            documents=product_documents[i:end],
            metadatas=product_metadata[i:end],
            ids=product_ids[i:end]
        )

    # Ingredient collection
    try:
        chroma_client.delete_collection("skincare_ingredients")
    except:
        pass

    ingredient_collection = chroma_client.create_collection(
        name="skincare_ingredients",
        metadata={"hnsw:space": "cosine"}
    )

    ingredient_ids = []
    idx = 0
    while idx < len(inci_df):
        ingredient_ids.append("ingredient_" + str(idx))
        idx = idx + 1

    ingredient_metadata = []
    for idx, row in inci_df.iterrows():
        ingredient_metadata.append({
            "name": str(row["name"]) if pd.notna(row["name"]) else "",
            "good_for": str(row["good_for"]),
            "avoid_for": str(row["avoid_for"])
        })

    ingredient_collection.add(
        documents=inci_df["full_description"].tolist(),
        metadatas=ingredient_metadata,
        ids=ingredient_ids
    )

    print(f"Indexed {product_collection.count()} products and {ingredient_collection.count()} ingredients.")
    return product_collection, ingredient_collection

Overwriting /content/drive/MyDrive/skincare_project/data_loader.py


## agents/skin_profile.py
Agent 1: Takes raw user text and extracts a structured skin profile using Gemini.

In [14]:
%%writefile /content/drive/MyDrive/skincare_project/agents/skin_profile.py
import json
import re
import base64


def load_image_base64(image_path):
    """Load an image from disk and encode it as base64."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def skin_profile_agent(client, user_input, image_path=None):
    """Parse user input (and optional facial image) into a structured skin profile.

    Args:
        client: Google GenAI client instance.
        user_input: Raw natural language string from the user.
        image_path: Optional path to a facial image (jpg/png) for visual analysis.

    Returns:
        dict with keys: skin_type, concerns, allergies, age, current_products,
        goals, routine_request. Visual observations from the image are merged
        into concerns if an image is provided.
    """

    text_prompt = """You are a skincare specialist. Extract a structured skin profile from the user input below.
If a facial image is also provided, analyze it for visible skin conditions such as acne, redness, pigmentation, or under-eye bags, and incorporate these observations into the profile — even if the user did not mention them in text.

Return ONLY valid JSON with these exact fields:
{
  "skin_type": "oily" | "dry" | "combination" | "normal" | "sensitive" | null,
  "concerns": ["acne", "redness", "dark spots", ...],
  "allergies": ["fragrance", "parabens", ...],
  "age": number or null,
  "current_products": ["product name", ...],
  "goals": ["hydration", "anti-aging", ...],
  "routine_request": "full routine" | "morning only" | "evening only" | "product suggestion" | null,
  "image_observations": "brief description of visible skin conditions detected from image, or null if no image"
}

Rules:
- skin_type: infer from user description or image if possible, else null
- concerns: combine user-reported concerns with any conditions visibly detected in the image
- allergies: only from user text, never infer from image
- image_observations: fill this field only if an image was provided, summarize what you see
- Return null for fields you cannot determine
- No markdown, no backticks, just the JSON

USER INPUT: """ + user_input

    try:
        if image_path:
            # Load and encode image
            image_data = load_image_base64(image_path)

            # Determine media type from extension
            ext = image_path.lower().split(".")[-1]
            media_type = "image/jpeg" if ext in ["jpg", "jpeg"] else "image/png"

            # Send text + image to Gemini
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    {
                        "parts": [
                            {"text": text_prompt},
                            {
                                "inline_data": {
                                    "mime_type": media_type,
                                    "data": image_data
                                }
                            }
                        ]
                    }
                ]
            )
        else:
            # Text only
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=text_prompt
            )

        # Parse JSON response
        text = re.sub(r'```json|```', '', response.text.strip()).strip()
        profile = json.loads(text)

        # Ensure all expected fields exist with safe defaults
        profile.setdefault("skin_type", None)
        profile.setdefault("concerns", [])
        profile.setdefault("allergies", [])
        profile.setdefault("age", None)
        profile.setdefault("current_products", [])
        profile.setdefault("goals", [])
        profile.setdefault("routine_request", None)
        profile.setdefault("image_observations", None)

        # Normalize None values
        if not profile["concerns"]:
            profile["concerns"] = []
        if not profile["allergies"]:
            profile["allergies"] = []
        if not profile["goals"]:
            profile["goals"] = []
        if not profile["current_products"]:
            profile["current_products"] = []

        # Print image observations if detected
        if profile.get("image_observations"):
            print(f"[skin_profile] Image analysis: {profile['image_observations']}")

        return profile

    except json.JSONDecodeError as e:
        print(f"[skin_profile] JSON parse failed: {e}")
        return {
            "skin_type": None, "concerns": [], "allergies": [],
            "age": None, "current_products": [], "goals": [],
            "routine_request": None, "image_observations": None
        }
    except Exception as e:
        print(f"[skin_profile] Agent failed: {e}")
        return {
            "skin_type": None, "concerns": [], "allergies": [],
            "age": None, "current_products": [], "goals": [],
            "routine_request": None, "image_observations": None
        }

Overwriting /content/drive/MyDrive/skincare_project/agents/skin_profile.py


## agents/retrieval.py
Agent 2: Searches the ChromaDB product collection using natural language queries.

In [15]:
%%writefile /content/drive/MyDrive/skincare_project/agents/retrieval.py
import json
import re

def search_products(product_collection, query, n_results=5, product_type=None, skin_type=None):
    """Search the product vector database using a natural language query.

    Args:
        product_collection: ChromaDB collection of skincare products.
        query: Natural language search string.
        n_results: Number of products to return.
        product_type: Optional filter (e.g., 'Cleanser', 'Moisturizer').
        skin_type: Optional skin type to filter results (e.g., 'oily', 'dry').

    Returns:
        ChromaDB query results dict with ids, documents, metadatas, distances.
    """
    where_filter = {"product_type": product_type} if product_type else None

    # Fetch more candidates upfront so we have enough after skin type filtering
    fetch_n = n_results * 3 if skin_type else n_results

    results = product_collection.query(
        query_texts=[query],
        n_results=fetch_n,
        where=where_filter
    )

    if not skin_type:
        # Trim to n_results
        return {
            "ids": [results["ids"][0][:n_results]],
            "documents": [results["documents"][0][:n_results]],
            "metadatas": [results["metadatas"][0][:n_results]],
            "distances": [results["distances"][0][:n_results]]
        }

    # Post-filter by skin type compatibility
    skin_type_lower = skin_type.lower().strip()
    filtered = {"ids": [], "documents": [], "metadatas": [], "distances": []}
    fallback = {"ids": [], "documents": [], "metadatas": [], "distances": []}

    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        st = meta.get("skin_types", "not specified").lower()
        entry = {
            "id": results["ids"][0][i],
            "doc": results["documents"][0][i],
            "meta": meta,
            "dist": results["distances"][0][i]
        }
        if st == "not specified" or skin_type_lower in st:
            filtered["ids"].append(entry["id"])
            filtered["documents"].append(entry["doc"])
            filtered["metadatas"].append(entry["meta"])
            filtered["distances"].append(entry["dist"])
        else:
            fallback["ids"].append(entry["id"])
            fallback["documents"].append(entry["doc"])
            fallback["metadatas"].append(entry["meta"])
            fallback["distances"].append(entry["dist"])

        if len(filtered["ids"]) >= n_results:
            break

    # If not enough compatible products, pad with best remaining results
    if len(filtered["ids"]) < n_results:
        needed = n_results - len(filtered["ids"])
        filtered["ids"] += fallback["ids"][:needed]
        filtered["documents"] += fallback["documents"][:needed]
        filtered["metadatas"] += fallback["metadatas"][:needed]
        filtered["distances"] += fallback["distances"][:needed]

    return {
        "ids": [filtered["ids"]],
        "documents": [filtered["documents"]],
        "metadatas": [filtered["metadatas"]],
        "distances": [filtered["distances"]]
    }

Overwriting /content/drive/MyDrive/skincare_project/agents/retrieval.py


## agents/conflict_checker.py
Agent 3: Checks for ingredient conflicts and beneficial pairs across retrieved products.

In [16]:
%%writefile /content/drive/MyDrive/skincare_project/agents/conflict_checker.py
import json
import re

# Ingredient interaction rules grounded in the Skincare Knowledge Graph dataset
# Rules are split into two categories:
#   - conflict_rules_raw: ingredient combinations to AVOID (adverse effects)
#   - beneficial_pairs: ingredient combinations that WORK WELL together

conflict_rules_raw = [
    {"ingredient_a": "retinol", "ingredient_b": "vitamin c", "reason": "Cancel out effects."},
    {"ingredient_a": "retinol", "ingredient_b": "aha", "reason": "Cancel out effects and cause irritation."},
    {"ingredient_a": "retinol", "ingredient_b": "bha", "reason": "May cause breakouts, dry skin, and irritation."},
    {"ingredient_a": "retinol", "ingredient_b": "benzoyl peroxide", "reason": "Too harsh for skin and cancel out effects."},
    {"ingredient_a": "salicylic acid", "ingredient_b": "benzoyl peroxide", "reason": "Can cause skin irritation."},
]

beneficial_pairs = [
    {"ingredient_a": "hyaluronic acid", "ingredient_b": "polyglutamic acid", "benefit": "Better hydration."},
    {"ingredient_a": "retinol", "ingredient_b": "niacinamide", "benefit": "Improves skin blemishes, diminishes ageing, and evens out skin tone."},
    {"ingredient_a": "retinol", "ingredient_b": "peptides", "benefit": "Produces collagen and hyaluronic acid, reduces inflammation, and increases cell turnover."},
    {"ingredient_a": "vitamin c", "ingredient_b": "vitamin e", "benefit": "Can help prevent photodamage."},
    {"ingredient_a": "vitamin c", "ingredient_b": "ferulic acid", "benefit": "Ferulic acid stabilizes vitamin C and fends off free radicals."},
]

# Build bidirectional conflict lookup so checking either ingredient finds the rule
conflict_lookup = {}
for rule in conflict_rules_raw:
    a, b = rule["ingredient_a"], rule["ingredient_b"]
    conflict_lookup.setdefault(a, []).append({"conflicts_with": b, "reason": rule["reason"]})
    conflict_lookup.setdefault(b, []).append({"conflicts_with": a, "reason": rule["reason"]})

# Build bidirectional beneficial lookup similarly
beneficial_lookup = {}
for pair in beneficial_pairs:
    a, b = pair["ingredient_a"], pair["ingredient_b"]
    beneficial_lookup.setdefault(a, []).append({"pairs_with": b, "benefit": pair["benefit"]})
    beneficial_lookup.setdefault(b, []).append({"pairs_with": a, "benefit": pair["benefit"]})

# Alias mapping: generic terms → actual INCI ingredient name variants
INGREDIENT_ALIASES = {
    "aha": ["glycolic acid", "lactic acid", "mandelic acid"],
    "bha": ["salicylic acid", "beta hydroxy acid"],
    "vitamin c": ["ascorbic acid", "l-ascorbic acid", "sodium ascorbyl phosphate",
                  "ascorbyl glucoside", "ascorbyl tetraisopalmitate", "3-o-ethyl ascorbic acid"],
    "retinol": ["retinol", "retinyl palmitate", "retinyl acetate", "retinal",
                "hydroxypinacolone retinoate"],
    "benzoyl peroxide": ["benzoyl peroxide"],
    "niacinamide": ["niacinamide", "nicotinamide"],
}

def normalize_ingredient(ing):
    """Map an ingredient name to its canonical conflict_lookup key if an alias exists."""
    ing_lower = ing.lower().strip()
    for canonical, aliases in INGREDIENT_ALIASES.items():
        if ing_lower == canonical or ing_lower in aliases:
            return canonical
    return ing_lower

def rag_conflict_check(client, ingredient_collection, ingredients_list):
    uncovered = [i for i in ingredients_list if i not in conflict_lookup]
    if not uncovered:
        return []

    # Query ChromaDB per ingredient for better retrieval
    all_docs = []
    for ing in uncovered[:15]:
        res = ingredient_collection.query(query_texts=[ing], n_results=3)
        if res["documents"][0]:
            all_docs.extend(res["documents"][0])

    context = "\n\n".join(all_docs)
    if not context.strip():
        return []

    prompt = f"""You are a cosmetic chemist and dermatologist reviewing a MULTI-PRODUCT skincare routine.
The ingredients below come from DIFFERENT products assigned to DIFFERENT routine steps (cleanser, serum, moisturizer, etc.).
The routine builder will AUTOMATICALLY separate conflicting ingredients across AM and PM routines.

Flag a conflict ONLY if ALL of the following are true:
1. A dermatologist would tell a patient to NEVER have both products in their routine on the same day, even at different times.
2. The harm is documented and clinically significant — causes real skin damage, not just reduced efficacy.
3. The interaction is well-established in skincare literature, not theoretical.

DO NOT flag:
- Ingredients that are simply suboptimal or slightly less effective together
- Combinations that are fine when used at different times of day (AM vs PM)
- AHA + BHA combinations — these are commonly used in full routines with AM/PM separation
- Formulation-level incompatibilities (e.g. cationic/anionic ingredient reactions that only matter inside a bottle during manufacturing)
- Ingredients you are uncertain about
- Any combination that is merely "less than ideal" rather than genuinely harmful
- Two AHA ingredients (e.g. glycolic acid + lactic acid) from different products — manageable with routine ordering
- Alcohol denat combined with AHA or BHA — not a clinically significant same-day hazard
- Any combination that the routine builder can resolve by separating into AM vs PM

FULL ROUTINE INGREDIENTS: {ingredients_list}
INGREDIENTS TO FOCUS ON (not covered by hardcoded rules): {uncovered[:15]}

INCI DESCRIPTIONS:
{context}

Return ONLY a valid JSON list of SEVERE, clinically documented, same-day conflicts. Each item must have:
- ingredient_a
- ingredient_b
- reason
- severity: must be "severe"

If uncertain or if no same-day severe conflicts exist, return []. No markdown, no explanation, just the JSON list."""

    try:
        response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        text = re.sub(r'```json|```', '', response.text.strip()).strip()
        rag_conflicts = json.loads(text)

        # Safety filter: drop anything not explicitly labeled severe
        rag_conflicts = [c for c in rag_conflicts if c.get("severity", "").lower() == "severe"]

        # Deduplicate (a,b) and (b,a) pairs
        seen = set()
        deduped = []
        for c in rag_conflicts:
            key = frozenset([c["ingredient_a"], c["ingredient_b"]])
            if key not in seen:
                seen.add(key)
                c["source"] = "rag"
                deduped.append(c)

        return deduped

    except json.JSONDecodeError as e:
        print(f"[conflict_checker] Failed to parse Gemini response: {e}")
        return []
    except Exception as e:
        print(f"[conflict_checker] RAG check failed: {e}")
        return []


Overwriting /content/drive/MyDrive/skincare_project/agents/conflict_checker.py


## agents/budget_agent.py
Agent 4: Parses user input into a structured budget profile (overall limit, tier, per-category limits) and filters retrieved products to match the user's budget constraints.

In [17]:
%%writefile /content/drive/MyDrive/skincare_project/agents/budget_agent.py
import json
import re

PRODUCT_TYPES = ["Cleanser", "Serum", "Moisturizer", "Sun protect", "Retinol", "Face oil"]


def budget_agent(client, user_input):
    """Parse user input into a structured budget profile.

    Returns:
        dict with keys:
            - overall_limit: float or None
            - tier: "under_50" / "under_100" / "under_200" / "200_plus" / "any"
            - per_category: dict mapping product type to float limit or None
            - raw_mention: the budget-related phrase the user said, or None
    """

    prompt = """You are a skincare budget specialist. Extract budget information from the user input.

User input: \"""" + user_input + """\"

Product categories to consider: Cleanser, Serum, Moisturizer, Sun protect, Retinol, Face oil.
(Note: "sunscreen" or "SPF" maps to Sun protect.)

EXTRACT THESE FIELDS:

1. overall_limit: total budget in dollars as a number, or null if not mentioned.
   Examples:
     "I have $150 to spend total"           -> 150
     "keep the whole routine under $80"     -> 80
     "no budget for the whole thing"        -> null (this is a 200_plus tier signal, not a number)

2. tier: classify the OVERALL budget into one value:
     under_50  -> cheap, budget, drugstore, affordable, under $50
     under_100 -> mid-range, moderate, under $100
     under_200 -> under $200
     200_plus  -> luxury, high-end, splurge, no budget limit, $200+
     any       -> no budget mentioned at all

3. per_category: a dict where each key is a product type. For each one,
   extract a dollar limit IF AND ONLY IF the user mentioned a specific
   amount tied to that product. Otherwise null.

   Catch these phrasings:
     "I'd spend $60 on a serum"                       -> Serum: 60
     "willing to splurge on moisturizer up to $120"   -> Moisturizer: 120
     "cleanser should be cheap, like $15 max"         -> Cleanser: 15
     "keep sunscreen under $25"                       -> Sun protect: 25
     "retinol can be expensive, $80 ok"               -> Retinol: 80
     "face oil no more than $40"                      -> Face oil: 40
     "$50 for cleanser, $70 for serum, $100 for moisturizer" -> Cleanser: 50, Serum: 70, Moisturizer: 100
     "spend more on serum and moisturizer, less on cleanser" -> null for all (no numbers given)
     "drugstore cleanser but high-end serum"          -> Cleanser: 50, Serum: null (qualitative only on serum)

   IMPORTANT: only fill a per_category value when there is a NUMERIC dollar
   amount tied to that specific product type. If the user only gives a
   qualitative preference (cheap, expensive, splurge) without a number
   for that category, leave it as null and let the tier handle it.

4. raw_mention: copy the exact budget-related phrase from the user, or null.

If the user mentions no budget at all, return tier "any" and everything else null.

Return ONLY valid JSON, no markdown, no backticks, no explanation:
{
    "overall_limit": null,
    "tier": "any",
    "per_category": {
        "Cleanser": null,
        "Serum": null,
        "Moisturizer": null,
        "Sun protect": null,
        "Retinol": null,
        "Face oil": null
    },
    "raw_mention": null
}"""

    try:
        response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        text = re.sub(r'```json|```', '', response.text.strip()).strip()
        budget_profile = json.loads(text)
    except json.JSONDecodeError:
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            try:
                budget_profile = json.loads(json_match.group())
            except json.JSONDecodeError:
                budget_profile = _default_budget_profile()
        else:
            budget_profile = _default_budget_profile()
    except Exception as e:
        print("[budget_agent] Gemini call failed: " + str(e))
        budget_profile = _default_budget_profile()

    budget_profile = _validate_budget_profile(budget_profile)
    return budget_profile


def _default_budget_profile():
    """Return a safe default profile with no budget constraint."""
    return {
        "overall_limit": None,
        "tier": "any",
        "per_category": {ptype: None for ptype in PRODUCT_TYPES},
        "raw_mention": None
    }


def _validate_budget_profile(profile):
    """Ensure all expected keys exist and values are the right types."""
    defaults = _default_budget_profile()

    for key in defaults:
        if key not in profile:
            profile[key] = defaults[key]

    # Fill any missing product types in per_category
    for ptype in PRODUCT_TYPES:
        if ptype not in profile["per_category"]:
            profile["per_category"][ptype] = None

    # Coerce overall_limit to float
    try:
        if profile["overall_limit"] is not None:
            profile["overall_limit"] = float(profile["overall_limit"])
    except (ValueError, TypeError):
        profile["overall_limit"] = None

    # Coerce per-category limits to float
    for ptype in profile["per_category"]:
        try:
            if profile["per_category"][ptype] is not None:
                profile["per_category"][ptype] = float(profile["per_category"][ptype])
        except (ValueError, TypeError):
            profile["per_category"][ptype] = None

    # Validate tier value
    valid_tiers = ["under_50", "under_100", "under_200", "200_plus", "any"]
    if profile["tier"] not in valid_tiers:
        profile["tier"] = "any"

    return profile


def filter_products_by_budget(products, budget_profile):
    """Filter products using per-category limits with overall limit as fallback.

    Priority order for each product:
      1. Per-category limit if set for that product type
      2. Overall limit if set
      3. Tier-derived limit if overall_limit is null
      4. No filter if tier is 'any' and nothing else is set

    Falls back to cheapest available option if a product type is
    wiped out entirely after filtering, so the routine is never incomplete.
    """

    # Short circuit when no budget mentioned at all
    if budget_profile["tier"] == "any" and budget_profile["overall_limit"] is None:
        has_any_cat_limit = any(v is not None for v in budget_profile["per_category"].values())
        if not has_any_cat_limit:
            return products

    TIER_LIMITS = {
        "under_50":  50.0,
        "under_100": 100.0,
        "under_200": 200.0,
        "200_plus":  None,
        "any":       None,
    }

    tier_limit    = TIER_LIMITS.get(budget_profile["tier"])
    overall_limit = budget_profile["overall_limit"]
    per_category  = budget_profile["per_category"]

    def get_limit_for_type(ptype):
        cat_limit = per_category.get(ptype)
        if cat_limit is not None:
            return cat_limit
        if overall_limit is not None:
            return overall_limit
        return tier_limit   # None means no cap

    def parse_price(price_str):
        try:
            return float(str(price_str).replace("$", "").replace(",", "").strip())
        except (ValueError, TypeError):
            return None

    def price_ok(p, limit):
        if limit is None:
            return True
        price = parse_price(p.get("price"))
        if price is None:
            return True   # unknown price, keep rather than silently drop
        if budget_profile["tier"] == "200_plus" and per_category.get(p["product_type"]) is None:
            return price >= 200.0
        return price <= limit

    filtered = []
    for p in products:
        limit = get_limit_for_type(p["product_type"])
        if price_ok(p, limit):
            filtered.append(p)

    # Safety fallback: if a product type lost all candidates, restore the cheapest one
    types_before = set(p["product_type"] for p in products)
    types_after  = set(p["product_type"] for p in filtered)
    missing      = types_before - types_after

    for ptype in missing:
        candidates = [p for p in products if p["product_type"] == ptype]
        candidates.sort(key=lambda p: parse_price(p.get("price")) or 9999)
        filtered.append(candidates[0])
        print("[budget_agent] No products in budget for " + ptype + " - added cheapest available as fallback.")

    return filtered


def format_budget_for_prompt(budget_profile):
    """Return a human-readable budget summary to inject into the routine builder prompt."""
    tier_labels = {
        "under_50":  "under $50 (budget-friendly / drugstore)",
        "under_100": "under $100 (mid-range)",
        "under_200": "under $200",
        "200_plus":  "$200+ (luxury / premium)",
        "any":       "no specific budget constraint",
    }

    lines = []

    if budget_profile["raw_mention"]:
        lines.append("User said: \"" + budget_profile["raw_mention"] + "\"")

    if budget_profile["overall_limit"] is not None:
        lines.append("Overall limit: $" + str(int(budget_profile["overall_limit"])))
    else:
        lines.append("Budget tier: " + tier_labels.get(budget_profile["tier"], "not specified"))

    cat_limits = [
        k + ": $" + str(int(v))
        for k, v in budget_profile["per_category"].items()
        if v is not None
    ]
    if cat_limits:
        lines.append("Per-category limits: " + ", ".join(cat_limits))

    return "\n".join(lines) if lines else "No budget constraint specified."

Overwriting /content/drive/MyDrive/skincare_project/agents/budget_agent.py


## agents/routine_builder.py
Agent 5: Assembles retrieved products into a structured skincare routine.

In [18]:
%%writefile /content/drive/MyDrive/skincare_project/agents/routine_builder.py
import json
import re
from agents.budget_agent import format_budget_for_prompt


def routine_builder_agent(client, profile, retrieved_products, conflict_report, routine_pref=None, budget_profile=None):
    """Assemble retrieved products into a structured skincare routine using Gemini.

    Args:
        client: Google GenAI client instance.
        profile: dict from skin_profile_agent (skin_type, concerns, allergies, etc.).
        retrieved_products: list of dicts with keys 'name', 'brand', 'product_type', 'ingredients', 'price'.
        conflict_report: dict with keys 'conflicts' (list) and 'allergy_flags' (list) from conflict checker.
        routine_pref: 'AM', 'PM', or 'both' from pipeline routine preference detection.
        budget_profile: dict from budget_agent with overall_limit, tier, per_category, raw_mention.

    Returns:
        dict with keys: morning_routine, evening_routine, notes, warnings.
    """

    # Format product list for the prompt
    product_list_str = ""
    for i, p in enumerate(retrieved_products, 1):
        product_list_str += f"{i}. {p['name']} (Brand: {p['brand']}, Type: {p['product_type']}, Price: {p['price']})\n"
        product_list_str += f"   Ingredients: {p['ingredients'][:200]}...\n"

    # Format conflict info
    conflict_str = "None detected." if not conflict_report["conflicts"] else ""
    for c in conflict_report["conflicts"]:
        conflict_str += f"- AVOID: {c['ingredient_a']} + {c['ingredient_b']} — {c['reason']}\n"

    allergy_str = "None detected." if not conflict_report["allergy_flags"] else ""
    for a in conflict_report["allergy_flags"]:
        allergy_str += f"- Product '{a['product']}' contains allergen: {a['allergen']}\n"

    beneficial_str = ""
    if conflict_report.get("beneficial"):
        for b in conflict_report["beneficial"]:
            beneficial_str += f"- {b['ingredient_a']} + {b['ingredient_b']}: {b['benefit']}\n"
    if not beneficial_str:
        beneficial_str = "None identified."

    # Format budget info
    budget_str = format_budget_for_prompt(budget_profile) if budget_profile else "No budget constraint specified."

    prompt = f"""You are a skincare routine specialist. Based on the user's skin profile,
retrieved product candidates, and ingredient safety analysis, build a personalized
morning and evening skincare routine.

USER PROFILE:
- Skin type: {profile.get('skin_type') or 'unknown'}
- Concerns: {', '.join(profile.get('concerns') or []) or 'none specified'}
- Allergies: {', '.join(profile.get('allergies') or []) or 'none'}
- Age: {profile.get('age') or 'not specified'}
- Goals: {', '.join(profile.get('goals') or []) or 'general skincare'}
- Routine request: {profile.get('routine_request') or 'full routine'}

BUDGET:
{budget_str}

AVAILABLE PRODUCTS:
{product_list_str}

INGREDIENT CONFLICTS DETECTED:
{conflict_str}

ALLERGY FLAGS:
{allergy_str}

BENEFICIAL COMBINATIONS:
{beneficial_str}

INSTRUCTIONS:
1. BUDGET IS THE TOP PRIORITY. The user cannot afford products outside their budget,
   so a routine they can't pay for is useless. NEVER select a product whose price
   exceeds the user's per-category limit (if set), overall limit (if set), or tier limit.
   If a category has no in-budget option, OMIT that step entirely and note it in warnings —
   do NOT recommend an over-budget product as a fallback. The user can decide whether to
   add that step later when they have more to spend.
2. Select appropriate products from the list above for a morning and evening routine,
   choosing the best skin-type and concern match WITHIN budget.
3. DO NOT include any products flagged for allergy conflicts.
4. If ingredient conflicts were detected, separate conflicting products into different routines
   (e.g., one in morning, one in evening) or exclude one.
5. Order products in standard skincare order, for example: cleanser → toner → serum/treatment → moisturizer → sunscreen (AM only).
6. Explain WHY each product was chosen based on the user's concerns, goals, AND how it fits the budget.

Return ONLY valid JSON with no markdown formatting, no backticks, no explanation outside the JSON.
Use this exact structure:
{{
    "morning_routine": [
        {{
            "step": 1,
            "product_type": "Cleanser",
            "product_name": "...",
            "brand": "...",
            "why": "brief explanation of why this product suits the user"
        }}
    ],
    "evening_routine": [
        {{
            "step": 1,
            "product_type": "Cleanser",
            "product_name": "...",
            "brand": "...",
            "why": "brief explanation"
        }}
    ],
    "notes": ["any general skincare tips relevant to this user's profile"],
    "warnings": ["any flagged conflicts or allergies the user should be aware of"]
}}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    try:
        routine = json.loads(response.text)
    except json.JSONDecodeError:
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            try:
                routine = json.loads(json_match.group())
            except json.JSONDecodeError:
                routine = {"error": "Could not parse routine", "raw": response.text}
        else:
            routine = {"error": "Could not parse routine", "raw": response.text}

    return routine

Overwriting /content/drive/MyDrive/skincare_project/agents/routine_builder.py


## pipeline.py
Chains all four agents together into a single `full_pipeline` function.


In [19]:
%%writefile /content/drive/MyDrive/skincare_project/pipeline.py
import json
import re
from collections import defaultdict
from agents.skin_profile import skin_profile_agent
from agents.retrieval import search_products
from agents.conflict_checker import conflict_lookup, beneficial_lookup, rag_conflict_check, normalize_ingredient
from agents.budget_agent import budget_agent, filter_products_by_budget
from agents.routine_builder import routine_builder_agent


def detect_routine_preference(client, user_input):
    """Use Gemini to detect if user wants AM, PM, or both routines."""

    prompt = f"""A user is asking for skincare advice. Determine if they want a morning routine, evening routine, or both.

USER INPUT: "{user_input}"

Rules:
- If they explicitly mention morning, daytime, AM, or waking up → return AM
- If they explicitly mention night, evening, PM, bedtime, or before bed → return PM
- If they mention both, neither, or just ask for a general/full routine → return both

Return ONLY one word: AM, PM, or both. Nothing else."""

    try:
        response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        result = response.text.strip()
        if result in ["AM", "PM", "both"]:
            return result
        return "both"
    except Exception as e:
        print(f"[detect_routine_preference] Gemini call failed: {e}")
        return "both"


def run_conflict_check(client, ingredient_collection, products, profile):
    conflicts_found = []
    beneficial_found = []
    allergy_flags = []

    all_ingredients = set()
    for p in products:
        for ing in p["ingredients"].lower().split(", "):
            all_ingredients.add(ing.strip())
    all_ingredients_list = list(all_ingredients)

    # Step 1: hardcoded rules (fast)
    for ing in all_ingredients_list:
        norm = normalize_ingredient(ing)
        if norm in conflict_lookup:
              for conflict in conflict_lookup[norm]:
                partner = conflict["conflicts_with"]
                partner_matches = any(normalize_ingredient(i) == partner for i in all_ingredients_list)
                if partner_matches:
                    pair = tuple(sorted([norm, partner]))
                    entry = {"ingredient_a": pair[0], "ingredient_b": pair[1],
                            "reason": conflict["reason"], "source": "rules"}
                    if entry not in conflicts_found:
                        conflicts_found.append(entry)

    # Step 2: RAG fallback for ingredients not in hardcoded rules
    rag_conflicts = rag_conflict_check(client, ingredient_collection, all_ingredients_list)
    for c in rag_conflicts:
        pair = tuple(sorted([c["ingredient_a"], c["ingredient_b"]]))
        entry = {"ingredient_a": pair[0], "ingredient_b": pair[1],
                 "reason": c["reason"], "source": "rag"}
        if entry not in conflicts_found:
            conflicts_found.append(entry)

    # Beneficial pairs
    for ing in all_ingredients_list:
        if ing in beneficial_lookup:
            for pair in beneficial_lookup[ing]:
                partner = pair["pairs_with"]
                if partner in all_ingredients:
                    sorted_pair = tuple(sorted([ing, partner]))
                    entry = {"ingredient_a": sorted_pair[0], "ingredient_b": sorted_pair[1],
                             "benefit": pair["benefit"]}
                    if entry not in beneficial_found:
                        beneficial_found.append(entry)

    # Allergy check
    ALLERGEN_SYNONYMS = {
    "parabens": ["methylparaben", "ethylparaben", "propylparaben", "butylparaben", "isobutylparaben", "isopropylparaben"],
    "sulfates": ["sodium lauryl sulfate", "sodium laureth sulfate", "ammonium lauryl sulfate", "ammonium laureth sulfate", "sls", "sles"],
    "fragrance": ["fragrance", "parfum", "perfume", "aroma", "limonene", "linalool", "citronellol", "geraniol", "eugenol"],
    "alcohol": ["alcohol denat", "alcohol denat.", "denatured alcohol", "ethanol", "sd alcohol", "isopropyl alcohol"],
    "silicones": ["dimethicone", "cyclopentasiloxane", "cyclohexasiloxane", "phenyl trimethicone", "trimethylsiloxysilicate"],
    "essential oils": ["tea tree oil", "lavender oil", "peppermint oil", "eucalyptus oil", "rosemary oil"],
    "formaldehyde": ["formaldehyde", "dmdm hydantoin", "imidazolidinyl urea", "diazolidinyl urea", "quaternium-15", "bronopol"],
}

    def expand_allergen(allergen):
        """Expand a user-reported allergen term into all its chemical name variants."""
        allergen_lower = allergen.lower().strip()
        for key, synonyms in ALLERGEN_SYNONYMS.items():
            if allergen_lower == key or allergen_lower in synonyms:
                return synonyms + [key]
        return [allergen_lower]

    user_allergies = [a.lower() for a in (profile.get("allergies") or [])]
    for p in products:
        ingredients_lower = p["ingredients"].lower()
        for allergen in user_allergies:
            expanded = expand_allergen(allergen)
            for term in expanded:
                if term in ingredients_lower:
                    allergy_flags.append({"product": p["name"], "allergen": allergen})
                    break  # avoid duplicate flags for the same product/allergen pair
    return {"conflicts": conflicts_found, "allergy_flags": allergy_flags, "beneficial": beneficial_found}


def full_pipeline(client, product_collection, ingredient_collection, user_input, products_per_type=3, image_path=None):
    """Run the full skincare recommendation pipeline.

    Agent 1 (Skin Profile) -> Agent 2 (Product Retrieval) -> Agent 3 (Conflict Check) ->
    Agent 4 (Budget) -> Agent 5 (Routine Builder)

    Args:
        client: Google GenAI client instance.
        product_collection: ChromaDB collection of skincare products.
        ingredient_collection: ChromaDB collection of INCI ingredients.
        user_input: Raw natural language string from the user.
        products_per_type: Number of products to retrieve per category (default 3).

    Returns:
        dict with keys: profile, retrieved_products, conflict_report, routine,
                        raw_input, routine_preference, budget_profile.
    """

    # Step 0: Detect routine preference and budget profile before any agent runs
    routine_pref = detect_routine_preference(client, user_input)
    budget_profile = budget_agent(client, user_input)

    print(f"Routine preference detected: {routine_pref}")
    print(f"Budget tier detected: {budget_profile['tier']}")
    if budget_profile["overall_limit"] is not None:
        print(f"Overall limit: ${budget_profile['overall_limit']}")
    per_cat = {k: v for k, v in budget_profile["per_category"].items() if v is not None}
    if per_cat:
        print(f"Per-category limits: {per_cat}")
    if budget_profile["tier"] == "any" and budget_profile["overall_limit"] is None and not per_cat:
        print("No budget constraint mentioned - returning all price ranges.")

    # Agent 1: Parse user input into structured skin profile
    profile = skin_profile_agent(client, user_input, image_path=image_path)

    # Agent 2: Retrieve candidate products based on routine preference
    am_types = ["Cleanser", "Serum", "Moisturizer", "Sun protect"]
    pm_types = ["Cleanser", "Serum", "Moisturizer", "Retinol", "Face oil"]

    if routine_pref == "AM":
        product_types = am_types
    elif routine_pref == "PM":
        product_types = pm_types
    else:
        product_types = list(set(am_types + pm_types))  # all unique types for both

    skin_type = profile.get("skin_type", "")
    concerns = " ".join(profile.get("concerns") or [])

    all_products = []
    for ptype in product_types:
        query = f"{skin_type} skin {concerns} {ptype.lower()}"
        # Fetch a wider net when budget is constrained so filtering still leaves enough candidates
        fetch_n = products_per_type * 3 if budget_profile["tier"] != "any" else products_per_type
        results = search_products(product_collection, query, n_results=fetch_n, product_type=ptype, skin_type=skin_type)
        for i in range(len(results["ids"][0])):
            meta = results["metadatas"][0][i]
            all_products.append({
                "name": meta["name"],
                "brand": meta["brand"],
                "product_type": meta["product_type"],
                "ingredients": results["documents"][0][i],
                "price": meta["price"],
                # Tag each product with its intended time of use
                "routine_time": "AM" if ptype == "Sun protect" else
                               "PM" if ptype in ["Retinol", "Face oil"] else "both"
            })

    # Agent 4: Strict budget filter (no over-budget fallback)
    all_products = filter_products_by_budget(all_products, budget_profile)

    # Surface omitted categories so routine_builder can warn the user
    omitted_categories = getattr(filter_products_by_budget, "last_omitted", [])
    if omitted_categories:
        budget_profile["omitted_categories"] = omitted_categories

    # Trim back to products_per_type per type so the prompt stays manageable
    type_counts = defaultdict(int)
    trimmed = []
    for p in all_products:
        if type_counts[p["product_type"]] < products_per_type:
            trimmed.append(p)
            type_counts[p["product_type"]] += 1
    all_products = trimmed

    # Agent 3: Check for conflicts and allergies
    conflict_report = run_conflict_check(client, ingredient_collection, all_products, profile)

    # Agent 5: Build the routine (passes routine_pref and budget_profile so Gemini respects both)
    routine = routine_builder_agent(
        client, profile, all_products, conflict_report,
        routine_pref=routine_pref, budget_profile=budget_profile
    )

    return {
        "raw_input": user_input,
        "profile": profile,
        "retrieved_products": all_products,
        "conflict_report": conflict_report,
        "routine": routine,
        "routine_preference": routine_pref,
        "budget_profile": budget_profile,
    }

Overwriting /content/drive/MyDrive/skincare_project/pipeline.py


## Verify Files Created

In [20]:
base = "/content/drive/MyDrive/skincare_project"
expected_files = [
    "data_loader.py",
    "pipeline.py",
    "agents/__init__.py",
    "agents/skin_profile.py",
    "agents/retrieval.py",
    "agents/conflict_checker.py",
    "agents/budget_agent.py",
    "agents/routine_builder.py",
]

print("File verification:")
for f in expected_files:
    path = os.path.join(base, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {status} - {f}")

print("\nAll modules built! Open demo.ipynb to run the pipeline.")

File verification:
  OK - data_loader.py
  OK - pipeline.py
  OK - agents/__init__.py
  OK - agents/skin_profile.py
  OK - agents/retrieval.py
  OK - agents/conflict_checker.py
  OK - agents/budget_agent.py
  OK - agents/routine_builder.py

All modules built! Open demo.ipynb to run the pipeline.
